# Поиск оптимальных подрешёток для раскраски $\mathbb{R}^4$

Основной сценарий: построение разбиения Вороного решётки и перебор подрешёток
(`find_optimal`). Теория и устройство структур — в `voronoi_info.ipynb` и `docs/article.tex`.

In [ ]:
# подключаем пакет из src/ (альтернатива: pip install -e .)
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np

from voronoi4d import (
    VoronoiPolyhedra,
    find_optimal,
    generate_grids,
    lll_reduce,
    min_max_det,
    plot_results,
)

## Примеры решёток

In [ ]:
grid = np.array([
    [2, 0, 0, 0],
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [1, 0, 0, 1]
], dtype=float)

grid_ = np.array([
    [2.15, 0, 0, 0],
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [1, 0, 0, 1]
], dtype=float)

grid1 = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
], dtype=float)

grid2 = np.array([
    [1, -1, 0, 0],
    [0, 1, -1, 0],
    [0, 0, 1, -1],
    [0.5, 0.5, 0.5, 0.5]
], dtype=float)

grid3 = np.array([
    [1, 1, 0, 0],
    [1, -1, 0, 0],
    [0, 1, -1, 0],
    [0, 0, 1, -1]
], dtype=float)

grid4 = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0.5, 0.5, 0.5, 0.5]
], dtype=float)

grid5 = np.array([
    [2 ** 0.5, 0., 0., 0.],
    [-1 / (2 ** 0.5), 6 ** 0.5 / 3, 0., 0.],
    [0., -(2 / 3) ** 0.5, 2 / ((3 ** 0.5) * 10), 0.],
    [0., 0., -(3 ** 0.5) / 2, (5 ** 0.5) / 20]
], dtype=float)

## Построение разбиения Вороного и поиск подрешёток

In [ ]:
grid = grid_

print(grid)

grid = lll_reduce(grid)

vor4 = VoronoiPolyhedra(grid)
vor4.build()

det_dist, det_center, det_mat = find_optimal(
    range(49, 50),       # диапазон определителей (количеств цветов)
    2,                   # limits: границы коэффициентов точек подрешётки
    grid,
    vor4,
    vor4.max_len,
    threshold=1.0,       # минимально допустимое нормированное расстояние d
)
print(det_dist, det_center, det_mat)

## Перебор случайных решёток

In [ ]:
list_grids = generate_grids(norm_limit=2.0, norm_factor=1.01, cos_limit=0.001, det_limit=3.0)
min_det, max_det = min_max_det(list_grids)
print(min_det, max_det)

In [ ]:
# Поиск для одного определителя (пример: det=49 — флагманский результат).
# С версии 1.1.0 параметр limits устарел (точный перебор кандидатов);
# полный диапазон 2..50 считается минуты (см. audit-data/campaign_a.py).
det_dist, det_center, det_mat = find_optimal(
    range(49, 50), None, grid, vor4, vor4.max_len, threshold=1.0,
)
det_dist


## Результаты: количество цветов $N$ от запрещённого расстояния $d$

In [ ]:
# Итоги допустимых раскрасок D4 (проверено в 1.1.0; см. RESULTS-2026-07-21.md):
# k=49 -> d=sqrt(7/6)=1.080123 — минимальное допустимое k для D4.
import matplotlib.pyplot as plt
ks = [49, 52, 56, 60]
ds = [1.080123, 1.0, 1.0, 1.0]
plt.scatter(ks, ds); plt.axhline(1.0, ls='--', c='gray')
plt.xlabel('число цветов k'); plt.ylabel('d = D/diam')
plt.title('D4: допустимые раскраски (d >= 1)')
plt.show()
